# DCDN — a synthetic check of the division primitive

This notebook does two things, and it is deliberately small enough to read in one sitting:

1. **It verifies the claim.** On a task whose target is a quotient, a DCDN block reaches a
   normalized MSE of about `2e-05` while a DCNv2-style cross network of the same shape stops
   at `6e-01` — worse than predicting the mean.
2. **It opens the box.** Because a one-layer DCDN block is a *closed-form Laurent polynomial*
   in its inputs, the trained weights can be read back as an exact symbolic expression. The
   last section prints that expression and renders it.

Runtime: about one minute on a GPU, a few minutes on CPU.


## 1. Why a cross network cannot divide

A DCNv2 cross layer computes

$$x_{i+1} = x_0 \odot W x_i + x_i .$$

Every coordinate of $x_{i+1}$ is a **polynomial** in the coordinates of $x_0$. Stacking $L$
such layers raises the degree to $L+1$, but the result stays a polynomial: the span is
$\mathrm{span}\{x^\alpha : |\alpha| \le L+1,\ \alpha \ge 0\}$, with non-negative exponents only.

The function $n/d$ is not in that span on any domain containing $d \to 0$, and no finite
polynomial approximates it uniformly there. A network can fit $n/d$ on a bounded box away
from zero — that is Weierstrass — but the fit is a local surface, not the quotient, and it
degrades as soon as the operand scale moves.

### The DCDN recurrence

DCDN adds one term:

$$x_{i+1} = x_i + x_0 \odot W_p(x_i) + \mathrm{inv}(x_0) \odot W_d(x_i)$$

where $\mathrm{inv}(x_0)_k = 1/x_{0,k}$ for coordinates $k$ declared as **denominators** by a
fixed mask, and $0$ otherwise. Two properties follow immediately.

**The span becomes Laurent.** With one layer, coordinate $i$ of the output is

$$x_{1,i} = x_i\,(1 + b^p_i) + x_i \sum_j W_p[i,j]\,x_j
          + m_i \frac{1}{x_i}\Big(b^d_i + \sum_j W_d[i,j]\,x_j\Big),$$

so the exponent $-1$ is reachable. A quotient $x_j/x_i$ is a **single weight**, not an
approximation: setting $W_d[i,j] = 1$ and everything else to zero computes it exactly.

**The mask is a semantic declaration, not a convenience.** Only coordinates a contract
marks as legitimate positive denominators are inverted. The reciprocal is floored at
`div_eps` and gated by a per-row validity tensor, so a zero or missing denominator produces
a declared fallback rather than an infinity.


## 2. Setup


In [ ]:
import math, sys
import torch
import torch.nn as nn

sys.path.insert(0, 'src')
from dcdn.core import DCDNCore

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print(f'torch {torch.__version__} on {DEV}')


## 3. The task

Six operands, drawn **log-uniformly over four decades** so the model sees the quotient at
many scales rather than on one narrow box. The target is $y = x_0 / x_1$; the remaining four
operands are pure distractors. Exactly one coordinate — $x_1$ — is declared a denominator.


In [ ]:
F, DEN = 6, 1
MASK = [i == DEN for i in range(F)]

def sample(n, lo=0.1, hi=100.0, seed=0):
    g = torch.Generator().manual_seed(seed)
    span = math.log(hi) - math.log(lo)
    x = torch.exp(torch.rand(n, F, generator=g) * span + math.log(lo))
    y = (x[:, 0] / x[:, 1]).unsqueeze(1)
    return x, y

x, y = sample(5)
print('operands\n', x.numpy().round(3))
print('target  ', y.squeeze().numpy().round(4))


## 4. Two models of the same shape

Both take the six operands, apply one interaction stack, and read out with a single linear
head. The only difference is whether the stack owns a reciprocal branch.


In [ ]:
class DCDNRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = DCDNCore(F, num_layers=1, denominator_mask=MASK, div_eps=1e-3,
                             product_init_scale=0.1, div_init_scale=0.1,
                             runtime_value_checks=False)
        self.head = nn.Linear(F, 1)
    def forward(self, x):
        return self.head(self.core(x))

class CrossRegressor(nn.Module):
    """DCNv2-style cross: x_{i+1} = x_0 * W(x_i) + x_i. Products only."""
    def __init__(self, layers=2):
        super().__init__()
        self.w = nn.ModuleList(nn.Linear(F, F) for _ in range(layers))
        self.head = nn.Linear(F, 1)
    def forward(self, x):
        h = x
        for layer in self.w:
            h = x * layer(h) + h
        return self.head(h)


## 5. Train

The loss is MSE divided by the variance of the target, so **1.0 is exactly the constant
predictor**. This is the same normalization the offline studies use, which makes the number
below directly comparable to those tables.


In [ ]:
def train(model, steps=2000, lr=3e-3, seed=1):
    model = model.to(DEV)
    xtr, ytr = sample(20000, seed=seed)
    xva, yva = sample(5000, seed=seed + 100)
    xtr, ytr, xva, yva = (t.to(DEV) for t in (xtr, ytr, xva, yva))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    var = ytr.var()
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        idx = torch.randint(0, len(xtr), (1024,), generator=g).to(DEV)
        opt.zero_grad()
        loss = ((model(xtr[idx]) - ytr[idx]) ** 2).mean() / var
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    with torch.no_grad():
        return float(((model(xva) - yva) ** 2).mean() / yva.var())

dcdn = DCDNRegressor(); n_dcdn = train(dcdn); dcdn = dcdn.cpu()
cross = CrossRegressor(); n_cross = train(cross); cross = cross.cpu()

print(f'DCDN   normalized MSE = {n_dcdn:.3e}')
print(f'Cross  normalized MSE = {n_cross:.3e}   (1.0 = constant predictor)')
print(f'ratio  = {n_cross / n_dcdn:,.0f}x')


Expected output — DCDN around `2e-05`, the cross network around `6e-01`, a gap of roughly
**four orders of magnitude**. The cross network is not merely worse: at `0.6` it explains
40% of the variance, which on this task is what a smooth surface fitted to a quotient buys.


## 6. Explainability — reading the closed form off the weights

This is the part a black-box model cannot offer. A one-layer DCDN block followed by a linear
head is *exactly*

$$\hat{y} = c + \sum_i h_i x_i (1 + b^p_i)
          + \sum_{i,j} h_i W_p[i,j]\, x_i x_j
          + \sum_{i \in \mathcal{D}} h_i b^d_i \frac{1}{x_i}
          + \sum_{i \in \mathcal{D}, j} h_i W_d[i,j]\, \frac{x_j}{x_i},$$

with $\mathcal{D}$ the declared denominator set. Every coefficient is a product of two learned
numbers, so the trained network can be printed as a symbolic rational function with no
attribution method, no gradients, and no approximation.


In [ ]:
def laurent_terms(model):
    """Exact monomial coefficients of a one-layer DCDN core plus linear head."""
    block = model.core.layers[0]
    Wp = block.product_block.weight.detach();  bp = block.product_block.bias.detach()
    Wd = block.division_block.weight.detach(); bd = block.division_block.bias.detach()
    h  = model.head.weight.detach()[0];        c  = float(model.head.bias.detach()[0])
    terms = []
    for i in range(F):
        terms.append((f'x{i}', float(h[i] * (1 + bp[i])), 'linear'))
        for j in range(F):
            terms.append((f'x{i}*x{j}', float(h[i] * Wp[i, j]), 'product'))
        if MASK[i]:
            terms.append((f'1/x{i}', float(h[i] * bd[i]), 'reciprocal'))
            for j in range(F):
                terms.append((f'x{j}/x{i}', float(h[i] * Wd[i, j]), 'ratio'))
    terms.sort(key=lambda t: -abs(t[1]))
    return terms, c

terms, constant = laurent_terms(dcdn)
print(f'constant = {constant:+.4f}')
for name, coef, kind in terms[:8]:
    print(f'  {coef:+.4f}  {name:<10} [{kind}]')


The leading term is `x0/x1` with a coefficient of about **+1.000**. The network did not
approximate the quotient — it *selected* it, and every remaining term is an order of
magnitude smaller.

### A readable rendering

Coefficients are grouped by monomial family and drawn on a signed axis, so the dominance of
the ratio term is visible rather than inferred from a column of numbers.


In [ ]:
from IPython.display import HTML, Math, display

PALETTE = {'ratio': '#eb6834', 'reciprocal': '#eda100',
           'product': '#1baf7a', 'linear': '#2a78d6'}

def render_terms(terms, constant, top=10, width=760):
    rows = terms[:top]
    scale = max(abs(c) for _, c, _ in rows) or 1.0
    rh, pad, label_w = 30, 18, 190
    bar_w = width - label_w - 90
    height = pad * 2 + rh * len(rows) + 54
    zero = label_w + bar_w / 2
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
           f'viewBox="0 0 {width} {height}" font-family="Inter,system-ui,sans-serif">',
           f'<rect width="{width}" height="{height}" rx="10" fill="#fcfcfb"/>',
           f'<text x="{pad}" y="26" font-size="14" font-weight="600" fill="#1a1a19">'
           f'Learned Laurent expansion &#183; top {len(rows)} terms</text>',
           f'<text x="{pad}" y="44" font-size="11" fill="#5c5b54">coefficient of each '
           f'monomial in the closed form of the trained network '
           f'(constant {constant:+.4f})</text>',
           f'<line x1="{zero}" y1="{pad+40}" x2="{zero}" y2="{height-16}" '
           f'stroke="#d8d7d0"/>']
    for k, (name, coef, kind) in enumerate(rows):
        y = pad + 50 + k * rh
        w = abs(coef) / scale * (bar_w / 2 - 6)
        x = zero if coef >= 0 else zero - w
        svg.append(f'<text x="{label_w-12}" y="{y+13}" font-size="12.5" fill="#1a1a19" '
                   f'text-anchor="end" font-family="ui-monospace,Menlo,monospace">{name}</text>')
        svg.append(f'<rect x="{x:.1f}" y="{y+2}" width="{max(w,1.5):.1f}" height="16" '
                   f'rx="3" fill="{PALETTE.get(kind, "#8b8a80")}"/>')
        lx, anchor = ((x + w + 8), 'start') if coef >= 0 else ((x - 8), 'end')
        svg.append(f'<text x="{lx:.1f}" y="{y+14}" font-size="11.5" font-weight="600" '
                   f'fill="#1a1a19" text-anchor="{anchor}">{coef:+.4f}</text>')
    lx = pad
    for kind, colour in PALETTE.items():
        svg.append(f'<rect x="{lx}" y="{height-24}" width="10" height="10" rx="2" fill="{colour}"/>')
        svg.append(f'<text x="{lx+15}" y="{height-15}" font-size="10.5" fill="#5c5b54">{kind}</text>')
        lx += 26 + len(kind) * 6.2
    return '\n'.join(svg) + '</svg>'

display(HTML(render_terms(terms, constant)))


In [ ]:
def as_latex(terms, constant, top=4, cutoff=1e-3):
    """The trained network as one rational expression."""
    kept = [(n, c) for n, c, _ in terms[:top] if abs(c) >= cutoff]
    parts = []
    for name, coef in kept:
        symbol = name.replace('*', r' \cdot ')
        parts.append(f'{coef:+.4f}' + r'\,' + symbol)
    return r'\hat{y} = ' + f'{constant:+.4f}' + ''.join(parts) + r' + \varepsilon'

display(Math(as_latex(terms, constant)))
largest = abs(terms[4][1]) if len(terms) > 4 else float('nan')
print(f'epsilon collects every remaining term; the largest of them is {largest:.4f}')


## 7. What this does and does not show

**Shows.** The reciprocal branch is used, it is used for the right pair of operands, and the
resulting function can be recovered exactly from the weights. On a task with a quotient in
it, and with training that spans four decades of operand scale, DCDN learns the quotient
rather than a surface that resembles it.

**Does not show.** That this transfers to real recommendation data. It does not — see
`docs/rq2_rq3.md`, where a reduced-input DCDN fails to reach the full-input field on three of
four public datasets. This notebook establishes the *mechanism*; the datasets establish where
the mechanism pays and where it does not.

Nor does it show extrapolation. Every evaluation point here is drawn from the training
distribution. `docs/rq1_addenda.md` measures what happens outside it, and the answer is that
the head extrapolates badly once the operand scale moves beyond the trained range.
